# GenerativeAURORA — NSRDB Merge and Preprocessing

This notebook is the Jupyter/Google Colab version of `merge_nsrdb.py`. It combines the individual NSRDB CSV files produced by `download_nsrdb.py` into a single cleaned Parquet dataset for the GenerativeAURORA pipeline. The original script is designed as the **second data-processing step**, after the 25 site-year downloads. fileciteturn3file0L15-L45

## Notebook workflow

1. Import packages
2. Configure input/output paths
3. Validate the downloaded CSV files
4. Load and merge all CSV files
5. Ensure clearness index and sky regime are available
6. Build the datetime column
7. Filter to daytime observations
8. Inspect summary statistics
9. Save `nsrdb_merged.parquet`
10. Save `nsrdb_summary.json`
11. Proceed to `eda_nsrdb.py`

**Important:** This notebook expects the `nsrdb_data/` folder from the NSRDB download notebook/script to be available in the same Colab runtime or project directory.

## 1. Install and import required packages

`pandas` is used for tabular data processing and `pyarrow` is used by pandas to read/write Parquet files.

In [1]:
!pip -q install pandas pyarrow

import os
import glob
import json

import pandas as pd

print('Packages imported successfully.')

Packages imported successfully.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Configuration

The notebook expects the raw NSRDB CSV files in `nsrdb_data/` and writes the cleaned Parquet dataset and JSON summary to the current working directory.

In [3]:
# Directory containing the raw NSRDB CSV files.
DATA_DIR = '/content/drive/MyDrive/Colab Notebooks/nsrdb_data'

# Main cleaned dataset for downstream GenerativeAURORA processing.
OUTPUT_PARQUET = 'nsrdb_merged.parquet'

# Machine-readable summary of the merged dataset.
SUMMARY_FILE = 'nsrdb_summary.json'

# Expected study sites.
SITES = [
    'Phoenix_AZ',
    'LosAngeles_CA',
    'Denver_CO',
    'Miami_FL',
    'Seattle_WA',
]

# Study years.
YEARS = [2018, 2019, 2020, 2021, 2022]

EXPECTED_FILE_COUNT = len(SITES) * len(YEARS)

print(f'Input directory    : {DATA_DIR}')
print(f'Expected CSV files : {EXPECTED_FILE_COUNT}')
print(f'Output Parquet     : {OUTPUT_PARQUET}')
print(f'Summary JSON       : {SUMMARY_FILE}')

Input directory    : /content/drive/MyDrive/Colab Notebooks/nsrdb_data
Expected CSV files : 25
Output Parquet     : nsrdb_merged.parquet
Summary JSON       : nsrdb_summary.json


## 3. Validate the input files

Before merging, this cell checks whether `nsrdb_data/` exists and reports how many CSV files are available. The original workflow expects **25 files** (5 sites × 5 years).

In [4]:
def validate_input_files():
    """
    Check whether the expected NSRDB site-year files are available.

    Returns
    -------
    list
        Sorted list of CSV file paths found in DATA_DIR.

    Raises
    ------
    FileNotFoundError
        If DATA_DIR does not exist.
    """
    if not os.path.isdir(DATA_DIR):
        raise FileNotFoundError(
            f'Input directory not found: {DATA_DIR}'
        )

    files = sorted(glob.glob(os.path.join(DATA_DIR, '*.csv')))

    print('=== Input validation ===')
    print(f'Expected CSV files : {EXPECTED_FILE_COUNT}')
    print(f'Found CSV files    : {len(files)}')

    if len(files) != EXPECTED_FILE_COUNT:
        print(
            f'WARNING: Expected {EXPECTED_FILE_COUNT} files, '
            f'but found {len(files)}.'
        )
    else:
        print('File-count check: PASSED')

    return files


csv_files = validate_input_files()

=== Input validation ===
Expected CSV files : 25
Found CSV files    : 25
File-count check: PASSED


## 4. Load and merge all NSRDB CSV files

Each CSV is loaded separately, its column names are standardized to uppercase, and the files are concatenated into one DataFrame. This preserves the raw observations while making the column naming consistent for downstream processing.

In [5]:
def load_all_files(files):
    """
    Load and concatenate the NSRDB CSV files.

    Parameters
    ----------
    files : list
        List of CSV file paths to load.

    Returns
    -------
    pandas.DataFrame or None
        Merged dataset, or None if no files could be loaded.
    """
    print('=== Loading NSRDB files ===')
    dfs = []

    for f in files:
        try:
            df = pd.read_csv(f, encoding='utf-8')

            # Standardize column names.
            df.columns = (
                df.columns.astype(str)
                .str.strip()
                .str.upper()
            )

            fname = os.path.basename(f)
            print(
                f'Loaded {fname:<30} -> '
                f'{len(df):,} rows, {len(df.columns)} cols'
            )

            dfs.append(df)

        except Exception as exc:
            print(f'ERROR loading {f}: {exc}')

    if not dfs:
        print('No files could be loaded.')
        return None

    merged = pd.concat(dfs, ignore_index=True)

    print('')
    print(f'Total rows before filtering: {len(merged):,}')
    return merged


df = load_all_files(csv_files)

if df is None:
    raise RuntimeError('No NSRDB files were successfully loaded.')

=== Loading NSRDB files ===
Loaded Denver_CO_2018.csv             -> 8,760 rows, 21 cols
Loaded Denver_CO_2019.csv             -> 8,760 rows, 21 cols
Loaded Denver_CO_2020.csv             -> 8,784 rows, 21 cols
Loaded Denver_CO_2021.csv             -> 8,760 rows, 21 cols
Loaded Denver_CO_2022.csv             -> 8,760 rows, 21 cols
Loaded LosAngeles_CA_2018.csv         -> 8,760 rows, 21 cols
Loaded LosAngeles_CA_2019.csv         -> 8,760 rows, 21 cols
Loaded LosAngeles_CA_2020.csv         -> 8,784 rows, 21 cols
Loaded LosAngeles_CA_2021.csv         -> 8,760 rows, 21 cols
Loaded LosAngeles_CA_2022.csv         -> 8,760 rows, 21 cols
Loaded Miami_FL_2018.csv              -> 8,760 rows, 21 cols
Loaded Miami_FL_2019.csv              -> 8,760 rows, 21 cols
Loaded Miami_FL_2020.csv              -> 8,784 rows, 21 cols
Loaded Miami_FL_2021.csv              -> 8,760 rows, 21 cols
Loaded Miami_FL_2022.csv              -> 8,760 rows, 21 cols
Loaded Phoenix_AZ_2018.csv            -> 8,760 rows, 21 c

## 5. Ensure clearness index and sky-regime variables exist

The original workflow uses:

$$CI = \frac{GHI}{Clearsky\ GHI}$$

The clearness index is clipped to [0, 1]. Sky regime thresholds are:

- `CI ≤ 0.10` → Overcast
- `0.10 < CI ≤ 0.35` → Cloudy
- `CI > 0.35` → Clear

Existing `CLEARNESS_INDEX` and `SKY_REGIME` columns are preserved. If they are missing, they are calculated.

In [6]:
def ensure_regime_columns(df):
    """
    Ensure that clearness index and sky-regime variables are available.
    """
    if 'CLEARNESS_INDEX' not in df.columns:
        if 'GHI' in df.columns and 'CLEARSKY GHI' in df.columns:
            df['CLEARNESS_INDEX'] = (
                df['GHI']
                / df['CLEARSKY GHI'].replace(0, float('nan'))
            ).clip(0, 1).fillna(0)
        else:
            print(
                'WARNING: Cannot compute clearness index. '
                'GHI or CLEARSKY GHI missing.'
            )
            df['CLEARNESS_INDEX'] = 0.0

    if 'SKY_REGIME' not in df.columns:
        df['SKY_REGIME'] = 'Overcast'
        df.loc[df['CLEARNESS_INDEX'] > 0.10, 'SKY_REGIME'] = 'Cloudy'
        df.loc[df['CLEARNESS_INDEX'] > 0.35, 'SKY_REGIME'] = 'Clear'

    return df


df = ensure_regime_columns(df)
print('Regime columns are available.')

Regime columns are available.


## 6. Build the datetime column

The NSRDB time fields (`YEAR`, `MONTH`, `DAY`, `HOUR`) are combined into a single `DATETIME` column. The current workflow does not perform a timezone conversion.

In [7]:
def build_datetime_index(df):
    """
    Construct a datetime column from YEAR, MONTH, DAY, and HOUR.
    """
    time_cols = ['YEAR', 'MONTH', 'DAY', 'HOUR']

    if all(c in df.columns for c in time_cols):
        df['DATETIME'] = pd.to_datetime(
            df['YEAR'].astype(str) + '-'
            + df['MONTH'].astype(str).str.zfill(2) + '-'
            + df['DAY'].astype(str).str.zfill(2) + ' '
            + df['HOUR'].astype(str).str.zfill(2) + ':00:00'
        )
        print('Datetime column built from Year/Month/Day/Hour.')
    else:
        missing = [c for c in time_cols if c not in df.columns]
        print(
            'WARNING: Cannot build datetime. '
            f'Missing columns: {missing}'
        )

    return df


df = build_datetime_index(df)

Datetime column built from Year/Month/Day/Hour.


## 7. Filter to daytime observations

For the intended solar-generation modeling task, the current pipeline keeps observations satisfying `GHI > 0`. When solar zenith angle is available, it also requires `SOLAR ZENITH ANGLE < 90°`.

In [8]:
def filter_daytime(df):
    """
    Retain daytime observations for solar-generation modeling.
    """
    before = len(df)

    mask = df['GHI'] > 0

    if 'SOLAR ZENITH ANGLE' in df.columns:
        mask = mask & (df['SOLAR ZENITH ANGLE'] < 90)

    df_day = df[mask].copy()
    after = len(df_day)
    removed = before - after
    pct = round(100.0 * removed / before, 1) if before > 0 else 0

    print(
        f'Daytime filter: kept {after:,} rows, '
        f'removed {removed:,} nighttime rows ({pct}%)'
    )

    return df_day


df = filter_daytime(df)

Daytime filter: kept 110,502 rows, removed 108,618 nighttime rows (49.6%)


## 8. Inspect the merged daytime dataset

This diagnostic section prints the available columns, observation counts by site, GHI statistics, sky-regime distribution, and clearness-index statistics.

In [9]:
def print_summary(df):
    """Print descriptive statistics for the merged daytime dataset."""
    print('')
    print('=== Columns in merged dataset ===')
    print(list(df.columns))

    print('')
    print('=== Row count per site ===')
    if 'SITE_NAME' in df.columns:
        print(df.groupby('SITE_NAME').size().to_string())
    elif 'SITE' in df.columns:
        print(df.groupby('SITE').size().to_string())
    else:
        print('No SITE_NAME or SITE column found.')

    print('')
    print('=== GHI stats per site (daytime only, W/m2) ===')
    site_col = 'SITE_NAME' if 'SITE_NAME' in df.columns else 'SITE'
    if site_col in df.columns and 'GHI' in df.columns:
        stats = df.groupby(site_col)['GHI'].describe().round(1)
        print(stats.to_string())

    print('')
    print('=== Sky regime distribution per site ===')
    if site_col in df.columns and 'SKY_REGIME' in df.columns:
        regime_table = (
            df.groupby([site_col, 'SKY_REGIME'])
            .size()
            .unstack(fill_value=0)
        )
        print(regime_table.to_string())

    print('')
    print('=== Clearness index stats per site ===')
    if site_col in df.columns and 'CLEARNESS_INDEX' in df.columns:
        ci_stats = (
            df.groupby(site_col)['CLEARNESS_INDEX']
            .describe()
            .round(3)
        )
        print(ci_stats.to_string())


print_summary(df)


=== Columns in merged dataset ===
['YEAR', 'MONTH', 'DAY', 'HOUR', 'MINUTE', 'GHI', 'DHI', 'DNI', 'WIND SPEED', 'TEMPERATURE', 'RELATIVE HUMIDITY', 'PRESSURE', 'CLOUD TYPE', 'SOLAR ZENITH ANGLE', 'CLEARSKY GHI', 'SITE', 'CLIMATE', 'LAT', 'LON', 'CLEARNESS_INDEX', 'SKY_REGIME', 'DATETIME']

=== Row count per site ===
SITE
Denver_CO        22091
LosAngeles_CA    22123
Miami_FL         21972
Phoenix_AZ       22183
Seattle_WA       22133

=== GHI stats per site (daytime only, W/m2) ===
                 count   mean    std  min    25%    50%    75%     max
SITE                                                                  
Denver_CO      22091.0  402.2  289.9  1.0  137.0  365.0  631.0  1073.0
LosAngeles_CA  22123.0  452.8  304.3  2.0  178.0  438.0  697.0  1065.0
Miami_FL       21972.0  433.1  287.0  2.0  171.0  424.0  673.0  1049.0
Phoenix_AZ     22183.0  475.9  306.7  2.0  188.0  478.0  734.0  1098.0
Seattle_WA     22133.0  296.8  263.1  1.0   69.0  212.0  484.0   961.0

=== Sky regime

## 9. Save the cleaned dataset

The daytime merged dataset is saved as `nsrdb_merged.parquet`, which is the main input for the next EDA and sequence-construction stages.

In [10]:
df.to_parquet(OUTPUT_PARQUET, index=False)

print(f'Merged dataset saved to: {OUTPUT_PARQUET}')
print(f'Total rows (daytime): {len(df):,}')
print(f'Total columns       : {len(df.columns)}')

Merged dataset saved to: nsrdb_merged.parquet
Total rows (daytime): 110,502
Total columns       : 22


## 10. Save the JSON summary

The summary stores dataset size, columns, and site-level GHI, clearness-index, and sky-regime statistics.

In [11]:
def save_summary(df):
    """Save a machine-readable summary of the merged NSRDB dataset."""
    site_col = 'SITE_NAME' if 'SITE_NAME' in df.columns else 'SITE'

    summary = {
        'total_rows': len(df),
        'columns': list(df.columns),
        'sites': {},
    }

    if site_col in df.columns:
        for site, grp in df.groupby(site_col):
            regime_counts = {}

            if 'SKY_REGIME' in grp.columns:
                regime_counts = grp['SKY_REGIME'].value_counts().to_dict()

            summary['sites'][site] = {
                'rows': len(grp),
                'ghi_mean': (
                    round(float(grp['GHI'].mean()), 2)
                    if 'GHI' in grp.columns else None
                ),
                'ghi_max': (
                    round(float(grp['GHI'].max()), 2)
                    if 'GHI' in grp.columns else None
                ),
                'clearness_mean': (
                    round(float(grp['CLEARNESS_INDEX'].mean()), 3)
                    if 'CLEARNESS_INDEX' in grp.columns else None
                ),
                'sky_regime': regime_counts,
            }

    with open(SUMMARY_FILE, 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2)

    print(f'Summary saved to {SUMMARY_FILE}')


save_summary(df)

Summary saved to nsrdb_summary.json


## 11. Final verification

This cell confirms that the two expected outputs exist and shows their file sizes.

In [12]:
print('=== Final output check ===')

for filename in [OUTPUT_PARQUET, SUMMARY_FILE]:
    if os.path.exists(filename):
        size_mb = os.path.getsize(filename) / (1024 ** 2)
        print(f'[OK] {filename:<25} {size_mb:.2f} MB')
    else:
        print(f'[MISSING] {filename}')

print('')
print('Merge and preprocessing complete.')
print('Next step: run eda_nsrdb.py')


=== Final output check ===
[OK] nsrdb_merged.parquet      2.15 MB
[OK] nsrdb_summary.json        0.00 MB

Merge and preprocessing complete.
Next step: run eda_nsrdb.py
